# INGESTA DE DATOS A GRAN ESCALA

# Sobre Numpy y Pandas

NumPy (Numerical Python) es la piedra angular del ecosistema de análisis de datos en Python, diseñada para resolver la lentitud computacional de las listas nativas mediante el uso de estructuras de datos continuas en memoria denominadas <b>ndarrays</b> (arreglos N-dimensionales homogéneos). La virtud fundamental de NumPy radica en su capacidad para ejecutar operaciones vectorizadas y aplicar el mecanismo de broadcasting, lo que permite procesar bloques completos de datos mediante cómputo estructurado en C sin la necesidad de escribir bucles for explícitos. Es una herramienta indispensable en la ciencia de datos porque no solo ofrece un rendimiento numérico y una eficiencia de memoria órdenes de magnitud superiores a los tipos nativos de Python, sino que también sirve como el lenguaje de intercambio de datos universal sobre el cual están construidas librerías de más alto nivel como pandas, scikit-learn y SciPy. [+info Cap4 Mckinney](https://wesmckinney.com/book/numpy-basics)

Pandas es la biblioteca estándar para la manipulación y el análisis de datos estructurados en Python, diseñada para transformar el cómputo numérico de bajo nivel en un flujo de trabajo expresivo e intuitivo. Construida sobre la base de alto rendimiento de NumPy, la virtud principal de pandas reside en sus dos estructuras de datos fundamentales, <b>Series</b> y <b>DataFrame</b>, las cuales introducen etiquetado explícito (índices) para filas y columnas. Esto habilita capacidades críticas para la ciencia de datos como la alineación automática de datos en operaciones aritméticas, el tratamiento nativo e integrado de valores faltantes (NaN), el rebanado flexible mediante etiquetas (loc e iloc) y la ejecución rápida de estadísticas descriptivas. Es un componente indispensable en el ecosistema porque proporciona la capa de abstracción necesaria para limpiar, transformar, consultar y reorganizar datos relacionales y de series temporales antes de alimentarlos a modelos estadísticos o de aprendizaje automático. [+info Cap5 Mckinney](https://wesmckinney.com/book/pandas-basics)

In [1]:
import sys
sys.path.insert(0, "../src") #Agregar path a codigo fuente
%load_ext autoreload
%autoreload 2

In [2]:
import os
import urllib.request
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import gc
               
from utils import *

print("Librerías cargadas correctamente.")
print_ram_usage("Inicio de sesión")

Librerías cargadas correctamente.
[RAM Usage] Inicio de sesión: 136.41 MB


# Ingesta de datos en formato CSV

El método ```pd.read_csv()``` cuenta con parámetros para personalizar la lectura de datos, como la asignación de tipos (dtype), el parseo automático de fechas (parse_dates), el filtrado de columnas (usecols) y el manejo de valores nulos (na_values).

In [3]:
#Lectura directa
filepath_csv="../data/raw/viajes_demo.csv"
df_raw = pd.read_csv(filepath_csv, header=None, keep_default_na=False)
print(df_raw.info())
print(df_raw)
print(f"\nMemoria utilizada por el DataFrame: {df_raw.memory_usage(deep=True).sum()} bytes")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       6 non-null      object
 1   1       6 non-null      object
 2   2       6 non-null      object
 3   3       6 non-null      object
 4   4       6 non-null      object
 5   5       6 non-null      object
dtypes: object(6)
memory usage: 420.0+ bytes
None
          0                    1          2            3           4  \
0  id_viaje           fecha_hora  pasajeros  distancia_m  tarifa_usd   
1       101  2023-01-15 08:30:00          1          1.5         8.5   
2       102  2023-01-15 09:15:00          3          5.2        22.0   
3       103  2023-01-15 10:00:00                     0.8         5.0   
4       104  2023-01-15 11:20:00          2         12.4        45.0   
5       105  2023-01-15 12:05:00          5          3.1        14.5   

           5  
0  tipo_pago  
1    Tarjeta  
2   Efectivo

In [4]:
# Ingesta optimizada especificando parámetros de read_csv
df = pd.read_csv(
    filepath_csv,
    index_col="id_viaje",                         # Usar la columna id_viaje como variable llave
    parse_dates=["fecha_hora"],                    # Convertir a datetime64 nativo
    dtype={
        "pasajeros": "Int8",                      # Entero que admite valores nulos (Nullable Integer)
        "distancia_m": "float32",                 # Reducción de precisión para ahorrar RAM
        "tarifa_usd": "float32",
        "tipo_pago": "category"                   # Tipo categoría para columnas con pocos valores repetidos
    },
    usecols=["id_viaje","fecha_hora", "pasajeros", "distancia_m", "tarifa_usd", "tipo_pago"], # Elijo columnas de interes
    na_values=["N/A", ""]                         # Interpretación explícita de valores nulos
)

print("--- Dataframe Ingerido ---")
print(df)
print("\n--- Tipos de Datos Ingeridos ---")
print(df.dtypes)
print(f"\nMemoria utilizada por el DataFrame: {df.memory_usage(deep=True).sum()} bytes")

--- Dataframe Ingerido ---
                  fecha_hora  pasajeros  distancia_m  tarifa_usd tipo_pago
id_viaje                                                                  
101      2023-01-15 08:30:00          1          1.5         8.5   Tarjeta
102      2023-01-15 09:15:00          3          5.2        22.0  Efectivo
103      2023-01-15 10:00:00       <NA>          0.8         5.0   Tarjeta
104      2023-01-15 11:20:00          2         12.4        45.0       NaN
105      2023-01-15 12:05:00          5          3.1        14.5  Efectivo

--- Tipos de Datos Ingeridos ---
fecha_hora     datetime64[ns]
pasajeros                Int8
distancia_m           float32
tarifa_usd            float32
tipo_pago            category
dtype: object

Memoria utilizada por el DataFrame: 356 bytes


# Ingesta de datos en formato Excel

In [5]:
# Cargar el objeto ExcelFile para inspeccionar las hojas
filepath_xls="../data/raw/viajes_demo.xlsx"
xls_file = pd.ExcelFile(filepath_xls)
sheet_names = xls_file.sheet_names
if len(sheet_names) > 1:
    print(f"Múltiples hojas detectadas en **{filepath_xls}**.")
    print(f"Hojas disponibles: {sheet_names}")
    selected_sheet = sheet_names[0]
    print(f"Hoja elegida por defecto: {selected_sheet}")

message=detect_merged_cells(filename=filepath_xls, sheet_name=selected_sheet)
print(message)

df_raw = pd.read_excel(xls_file, sheet_name=selected_sheet, header=None, keep_default_na=False) # Leer la hoja específica conservando Logos/Metadata

print(df_raw.info())
print(df_raw)
print(f"\nMemoria utilizada por el DataFrame: {df_raw.memory_usage(deep=True).sum()} bytes")

Múltiples hojas detectadas en **../data/raw/viajes_demo.xlsx**.
Hojas disponibles: ['viajes_demo', 'Hoja2']
Hoja elegida por defecto: viajes_demo
No se encontraron celdas agrupadas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       6 non-null      object
 1   1       6 non-null      object
 2   2       6 non-null      object
 3   3       6 non-null      object
 4   4       6 non-null      object
 5   5       6 non-null      object
dtypes: object(6)
memory usage: 420.0+ bytes
None
          0                    1          2            3           4  \
0  id_viaje           fecha_hora  pasajeros  distancia_m  tarifa_usd   
1       101  2023-01-15 08:30:00          1          1.5         8.5   
2       102  2023-01-15 09:15:00          3          5.2          22   
3       103  2023-01-15 10:00:00                     0.8           5   
4       104  2023-01-

In [6]:
# Ingesta Optimizada especificando parámetros de pd.read_excel()

# A. Proyección: Seleccionar únicamente las columnas analíticas útiles (omitiendo 'notas_internas')
columnas_interes = ['id_viaje', 'fecha_hora', 'pasajeros', 'distancia_m']

# B. Definición de tipos de datos en la ingesta para minimizar memoria RAM
esquema_tipos = {
    'id_viaje': 'int32',
    'pasajeros': 'Int8',      # Entero nullable
    'distancia_m': 'float32'
}

# C. Carga optimizada especificando motor, hoja, proyección y tipos
df_excel = pd.read_excel(
    xls_file,
    sheet_name=selected_sheet,      
    usecols=columnas_interes,        # Proyección de columnas (Ahorro directo de memoria)
    dtype=esquema_tipos,             # Downcasting explícito de tipos de datos
    parse_dates=['fecha_hora'],      # Inferencia explícita de campos de fecha
    engine='openpyxl'                # Motor optimizado para archivos .xlsx
)

# D. Asignación de índice
df_excel = df_excel.set_index('id_viaje')

print("=== DATAFRAME INGERIDO DESDE EXCEL ===")
print(df_excel)

print("\n=== TIPOS DE DATOS Y USO DE MEMORIA ===")
print(df_excel.dtypes)
print(f"\nMemoria utilizada por el DataFrame: {df_excel.memory_usage(deep=True).sum()} bytes")

=== DATAFRAME INGERIDO DESDE EXCEL ===
                  fecha_hora  pasajeros  distancia_m
id_viaje                                            
101      2023-01-15 08:30:00          1          1.5
102      2023-01-15 09:15:00          3          5.2
103      2023-01-15 10:00:00       <NA>          0.8
104      2023-01-15 11:20:00          2         12.4
105      2023-01-15 12:05:00          5          3.1

=== TIPOS DE DATOS Y USO DE MEMORIA ===
fecha_hora     datetime64[ns]
pasajeros                Int8
distancia_m           float32
dtype: object

Memoria utilizada por el DataFrame: 90 bytes


# Consultas API REST

Una API REST es un estilo de arquitectura de software que permite la comunicación entre sistemas a través del protocolo HTTP.

REpresentational State Transfer (REST) basa la transferencia y manipulación de recursos (habitualmente en formato JSON) en los verbos estándar del protocolo HTTP:

Método HTTP|Acción|Uso común en APIs|
---|---|---|
GET|Lectura|Consultar u obtener información sin alterar datos en el servidor.|
POST|Creación|Enviar datos para registrar un nuevo recurso en el servidor.|
PUT / PATCH|Actualización|Modificar un recurso existente (de forma total con PUT o parcial con PATCH).|
DELETE|Eliminación|Borrar un recurso específico.|

Los servicios web poseen "endpoints" donde se pueden enviar mensajes en formato API REST.

<u>Principios fundamentales</u>:

- Sin estado (Stateless): Cada solicitud enviada al servidor contiene toda la información necesaria para ser procesada; el servidor no guarda contexto entre peticiones.

- Códigos de estado HTTP: Cada mensaje recibe una respuesta numérica estándar para indicar el resultado de la operación (200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, 404 Not Found, 500 Internal Server Error). [+ info sobre Códigos](https://http.cat/)

# La librería "Requests" de Python.

Conocida por su lema "HTTP for Humans", permite crear mensajes y comunicarse via API REST con una sintaxis limpia y declarativa.  [+ info sobre libreria requests](https://pypi.org/project/requests/)

<u>Funcionalidades</u>:

- Sintaxis directa: Métodos que coinciden con los verbos HTTP (requests.get(), requests.post(), etc.).

- Manejo automático de JSON: Convierte respuestas JSON directamente a diccionarios o listas de Python mediante el método .json().

- Autenticación y encabezados: Permite inyectar fácilmente headers (como tokens Bearer) y parámetros de consulta (query parameters).

- Control de respuestas: Acceso inmediato a códigos de estado (.status_code), encabezados (.headers) y manejo de excepciones (.raise_for_status()).

# Web scrapping

Dataset: 
- [NYC Yellow Taxi Trip Records](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) 
- [Diccionario](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

In [7]:
import requests
from bs4 import BeautifulSoup
import re

# 1. URL oficial del portal de datos de NYC TLC
url = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"

# Definir User-Agent para simular una petición desde un navegador web
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 2. Enviar la petición HTTP GET
response = requests.get(url, headers=headers)

if response.status_code == 200:
    # 3. Crear el objeto BeautifulSoup para parsear el HTML
    soup = BeautifulSoup(response.content, "html.parser")
    
    print("=" * 70)
    print("DESCRIPCIÓN GENERAL DEL DATASET (NYC TLC)")
    print("=" * 70)
    
    # Extraer el primer párrafo informativo dentro de la sección principal
    main_content = soup.find("div", class_="about-description") or soup.find("main")
    if main_content:
        paragraphs = main_content.find_all("p")
        for p in paragraphs[:3]:  # Obtener los primeros párrafos descriptivos
            text = p.get_text(strip=True)
            if text:
                print(f"\n• {text}")
    
    print("\n" + "=" * 70)
    print("ENLACES DE DESCARGA (YELLOW TAXI - 2023)")
    print("=" * 70)
    
    # 4. Extraer los enlaces .parquet de los Yellow Taxi Records
    # Se busca etiquetas <a> cuyo atributo 'href' contenga 'yellow_tripdata'
    yellow_links = soup.find_all("a", href=re.compile(r"yellow_tripdata_2023.*\.parquet"))
    
    for link in yellow_links[:5]:  # Mostrar los primeros 5 meses encontrados
        file_url = link['href']
        title = link.get_text(strip=True) or "Yellow Taxi Data"
        print(f"  ✓ {title}: {file_url}")

else:
    print(f"Error al acceder a la página. Código de estado: {response.status_code}")

C:\Users\Chewie\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


DESCRIPCIÓN GENERAL DEL DATASET (NYC TLC)

• Yellow and green taxi trip records include fields capturing pickup and drop-off dates/times, pickup and drop-off locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts. The data used in the attached datasets were collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers authorized under the Taxicab & Livery Passenger Enhancement Programs (TPEP/LPEP). The trip data was not created by the TLC, and TLC makes no representations as to the accuracy of these data.

• For-Hire Vehicle (“FHV”) trip records include fields capturing the dispatching base license number and the pickup date, time, and taxi zone location ID (shape file below).These records are generated from the FHV Trip Record submissions made by bases, so we cannot guarantee or confirm their accuracy or completeness. The TLC performs routine reviews of the records and takes enforcement actions when ne

# Sobre el formato Parquet

Apache Parquet es un formato de archivo de datos de código abierto orientado a columnas, diseñado para el almacenamiento y la recuperación eficiente de información. Ofrece esquemas de compresión y codificación de alto rendimiento para gestionar datos complejos a gran escala, y cuenta con soporte en múltiples lenguajes de programación y herramientas de analítica. Es el estándar de la industria para los data lakes modernos, arquitecturas de almacenamiento en la nube y herramientas de Big Data. [+ info sobre Parquet](https://parquet.apache.org/)

- Almacenamiento columnar: A diferencia de los formatos basados en filas (como CSV), Parquet almacena los datos columna por columna. Si una consulta solo solicita dos columnas de un total de cien, el motor de procesamiento evita físicamente leer el resto, lo que ahorra una cantidad significativa de memoria y capacidad de cómputo.

- Compresión avanzada: Debido a que los datos del mismo tipo se agrupan de forma contigua, los algoritmos de compresión operan con extrema eficiencia. Los datasets en Parquet suelen ser entre 5 y 10 veces más pequeños que sus equivalentes en archivos CSV, lo que reduce drásticamente los costos de almacenamiento en la nube.

- Autodescriptivo (Esquema integrado): Un archivo Parquet incluye sus propios metadatos, detallando con precisión qué columnas existen, sus nombres y sus tipos de datos. No es necesario parsear manualmente ni inyectar un esquema externo para poder leerlo.

- Metadatos y estadísticas: Los archivos Parquet conservan valores mínimos y máximos, conteo de nulos y total de filas dentro de su capa de pie de página (footer). Esto permite a los motores de consulta descartar bloques masivos de datos irrelevantes por completo ("predicate pushdown") sin necesidad de escanear las páginas de datos reales.

In [8]:
# Descargaremos 6 meses del dataset en formato Parquet
months = [f"2025-0{i}" for i in range(1, 9)]
files = []

print("Descargando archivos Parquet desde NYC TLC...")
for m in months:
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{m}.parquet"
    filename = f"yellow_tripdata_{m}.parquet"
    filepath="../data/raw/"+filename
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(url, filepath)
        print(f"  ✓ Descargado: {filename}")
    else:
        print(f"  ✓ Ya existe en disco: {filename}")
    files.append(filepath)
print("Descarga completada.\n")

Descargando archivos Parquet desde NYC TLC...
  ✓ Ya existe en disco: yellow_tripdata_2025-01.parquet
  ✓ Ya existe en disco: yellow_tripdata_2025-02.parquet
  ✓ Ya existe en disco: yellow_tripdata_2025-03.parquet
  ✓ Ya existe en disco: yellow_tripdata_2025-04.parquet
  ✓ Ya existe en disco: yellow_tripdata_2025-05.parquet
  ✓ Ya existe en disco: yellow_tripdata_2025-06.parquet
  ✓ Ya existe en disco: yellow_tripdata_2025-07.parquet
  ✓ Ya existe en disco: yellow_tripdata_2025-08.parquet
Descarga completada.



In [9]:
#Guardamos un mes en CSV para ver la diferencia
pd.read_parquet(files[0]).to_csv("../data/raw/demo.csv")

In [10]:
import time

parquet_file = files[0]
csv_file = "../data/raw/demo.csv"

# 1. Almacenamiento en disco
size_parquet_mb = os.path.getsize(parquet_file) / (1024 ** 2)
size_csv_mb = os.path.getsize(csv_file) / (1024 ** 2)

# 2. Velocidad de lectura completa
t0 = time.time()
df_csv = pd.read_csv(csv_file)
t_csv_full = time.time() - t0

t0 = time.time()
df_parquet = pd.read_parquet(parquet_file)
t_parquet_full = time.time() - t0

# 3. Velocidad de lectura parcial (Proyección de 2 columnas)
cols = ['tpep_pickup_datetime', 'trip_distance']

t0 = time.time()
df_csv_sub = pd.read_csv(csv_file, usecols=cols)
t_csv_sub = time.time() - t0

t0 = time.time()
df_parquet_sub = pd.read_parquet(parquet_file, columns=cols)
t_parquet_sub = time.time() - t0

# 4. Reporte de resultados
print(f"=== 1. TAMAÑO EN DISCO ===")
print(f"• Parquet : {size_parquet_mb:.2f} MB")
print(f"• CSV     : {size_csv_mb:.2f} MB")
print(f"--> Reducción: Parquet ocupa ~{((1 - size_parquet_mb/size_csv_mb) * 100):.1f}% menos espacio.\n")

print(f"=== 2. LECTURA COMPLETA ===")
print(f"• Parquet : {t_parquet_full:.2f} segundos")
print(f"• CSV     : {t_csv_full:.2f} segundos")
print(f"--> Desempeño: Parquet es ~{t_csv_full / t_parquet_full:.1f}x más rápido.\n")

print(f"=== 3. LECTURA PARCIAL (2 COLUMNAS) ===")
print(f"• Parquet : {t_parquet_sub:.2f} segundos")
print(f"• CSV     : {t_csv_sub:.2f} segundos")
print(f"--> Desempeño: Parquet es ~{t_csv_sub / t_parquet_sub:.1f}x más rápido al seleccionar columnas.\n")

print(f"=== 4. INTEGRIDAD DEL ESQUEMA ===")
print(f"• CSV Datetime    : {df_csv['tpep_pickup_datetime'].dtype} (Requiere parseo manual posterior)")
print(f"• Parquet Datetime: {df_parquet['tpep_pickup_datetime'].dtype} (Preserva tipo nativo)")

C:\Users\Chewie\AppData\Local\Temp\ipykernel_7080\518590899.py:12: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(csv_file)


=== 1. TAMAÑO EN DISCO ===
• Parquet : 56.42 MB
• CSV     : 388.09 MB
--> Reducción: Parquet ocupa ~85.5% menos espacio.

=== 2. LECTURA COMPLETA ===
• Parquet : 0.50 segundos
• CSV     : 7.76 segundos
--> Desempeño: Parquet es ~15.5x más rápido.

=== 3. LECTURA PARCIAL (2 COLUMNAS) ===
• Parquet : 0.12 segundos
• CSV     : 3.87 segundos
--> Desempeño: Parquet es ~32.5x más rápido al seleccionar columnas.

=== 4. INTEGRIDAD DEL ESQUEMA ===
• CSV Datetime    : object (Requiere parseo manual posterior)
• Parquet Datetime: datetime64[us] (Preserva tipo nativo)


In [11]:
# Cargar un mes directamente con pandas en su configuración por defecto
df_naive = pd.read_parquet(files[0])

mem_naive_bytes = df_naive.memory_usage(deep=True).sum()
mem_naive_mb = mem_naive_bytes / (1024 ** 2)

print(f"Filas cargadas: {len(df_naive):,}")
print(f"Columnas: {len(df_naive.columns)}")
print(f"Uso de RAM en DataFrame (Deep Memory): {mem_naive_mb:.2f} MB")
print("\nTipos de datos detectados por defecto:")
print(df_naive.dtypes)
print(df_naive.head())
print(df_naive.describe())
# Liberar memoria de la prueba
del df_naive
gc.collect()

Filas cargadas: 3,475,226
Columnas: 20
Uso de RAM en DataFrame (Deep Memory): 616.31 MB

Tipos de datos detectados por defecto:
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  pa

0

In [12]:
# ------------------------------------------------------------------------------
# Carga directa sin optimizar
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("DEMOSTRACIÓN DE CARGA TRADICIONAL (SIN OPTIMIZAR)")
print("="*60)

dfs = []

try:
    for f in files:
        ram_used = psutil.Process(os.getpid()).memory_info().rss / (1024**3)
        print(f"Cargando {f} | RAM en uso: {ram_used:.2f} GB")
        
        # Carga naive: 100% de columnas con tipos por defecto (float64, int64, object)
        df_month = pd.read_parquet(f)
        dfs.append(df_month)

    print("\nIntento de unificación masiva...")
    print("Ejecutando pd.concat() [Esto puede provocar un pico de memoria y el colapso del Kernel]...")
    
    # pd.concat debe asignar un nuevo bloque contiguo de RAM igual a la suma de todos los DataFrames
    df_6_months = pd.concat(dfs, ignore_index=True)
    
    print(f"Éxito. Filas totales: {len(df_6_months):,}")

except MemoryError:
    print("\n❌ FALLO DETECTADO: MemoryError atrapado por Python.")
except Exception as e:
    print(f"\n❌ FALLO GENERAL: {e}")

del df_6_months
gc.collect()


DEMOSTRACIÓN DE CARGA TRADICIONAL (SIN OPTIMIZAR)
Cargando ../data/raw/yellow_tripdata_2025-01.parquet | RAM en uso: 2.85 GB
Cargando ../data/raw/yellow_tripdata_2025-02.parquet | RAM en uso: 3.05 GB
Cargando ../data/raw/yellow_tripdata_2025-03.parquet | RAM en uso: 3.49 GB
Cargando ../data/raw/yellow_tripdata_2025-04.parquet | RAM en uso: 4.06 GB
Cargando ../data/raw/yellow_tripdata_2025-05.parquet | RAM en uso: 4.56 GB
Cargando ../data/raw/yellow_tripdata_2025-06.parquet | RAM en uso: 5.29 GB
Cargando ../data/raw/yellow_tripdata_2025-07.parquet | RAM en uso: 5.91 GB
Cargando ../data/raw/yellow_tripdata_2025-08.parquet | RAM en uso: 6.33 GB

Intento de unificación masiva...
Ejecutando pd.concat() [Esto puede provocar un pico de memoria y el colapso del Kernel]...
Éxito. Filas totales: 31,556,438


0

In [13]:
# ------------------------------------------------------------------------------
# ENFOQUE OPTIMIZADO: Proyección, Downcasting y Categorías
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("DEMOSTRACIÓN 2: INGESTA OPTIMIZADA")
print("="*60)

# A. Definir columnas necesarias (Proyección) para evitar cargar datos no usados
cols_to_use = [
    'tpep_pickup_datetime', 
    'tpep_dropoff_datetime', 
    'passenger_count', 
    'trip_distance', 
    'PULocationID', 
    'DOLocationID', 
    'payment_type', 
    'fare_amount', 
    'total_amount'
]

# B. Definir esquema con dtypes optimizados para minimizar huella en memoria
dtype_spec = {
    'passenger_count': 'float32',  # O Int8 si manejamos valores nulos correctamente
    'trip_distance': 'float32',
    'PULocationID': 'int16',
    'DOLocationID': 'int16',
    'payment_type': 'float32',    # Convertible a int8/category posteriormente
    'fare_amount': 'float32',
    'total_amount': 'float32'
}

# C. Carga en streaming usando PyArrow Dataset
dataset = ds.dataset(files, format="parquet")

# Convertir a Pandas cargando únicamente las columnas seleccionadas
df_opt = dataset.to_table(columns=cols_to_use).to_pandas()

# D. Aplicar Downcasting y Categorización explícita
for col, dtype in dtype_spec.items():
    if col in df_opt.columns:
        df_opt[col] = df_opt[col].astype(dtype)

# Convertir columnas con baja cardinalidad a 'category'
categorical_cols = ['PULocationID', 'DOLocationID', 'payment_type']
for col in categorical_cols:
    df_opt[col] = df_opt[col].astype('category')

# E. Medición de consumo en el DataFrame optimizado
mem_opt_bytes = df_opt.memory_usage(deep=True).sum()
mem_opt_mb = mem_opt_bytes / (1024 ** 2)

print(f"Filas cargadas (6 meses combinados): {len(df_opt):,}")
print(f"Columnas filtradas: {len(df_opt.columns)}")
print(f"Uso de RAM en DataFrame Optimizado: {mem_opt_mb:.2f} MB")
print(df_opt.head())
print(df_opt.info())


DEMOSTRACIÓN 2: INGESTA OPTIMIZADA
Filas cargadas (6 meses combinados): 31,556,438
Columnas filtradas: 9
Uso de RAM en DataFrame Optimizado: 1113.51 MB
  tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  \
0  2025-01-01 00:18:38   2025-01-01 00:26:59              1.0           1.60   
1  2025-01-01 00:32:40   2025-01-01 00:35:13              1.0           0.50   
2  2025-01-01 00:44:04   2025-01-01 00:46:01              1.0           0.60   
3  2025-01-01 00:14:27   2025-01-01 00:20:01              3.0           0.52   
4  2025-01-01 00:21:34   2025-01-01 00:25:06              3.0           0.66   

  PULocationID DOLocationID payment_type  fare_amount  total_amount  
0          229          237          1.0         10.0         18.00  
1          236          237          1.0          5.1         12.12  
2          141          141          1.0          5.1         12.10  
3          244          244          2.0          7.2          9.70  
4          244  

In [14]:
# ------------------------------------------------------------------------------
# COMPARACIÓN Y ANÁLISIS DE EFICIENCIA
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("RESUMEN DE DESEMPEÑO")
print("="*60)
# Proyectando el tamaño de 1 mes naive vs 1 mes optimizado
mem_single_month_opt_mb = df_opt.iloc[:3000000].memory_usage(deep=True).sum() / (1024**2)

print(f"• Consumo aproximado por mes (Enfoque Tradicional): ~{mem_naive_mb:.2f} MB")
print(f"• Consumo aproximado por mes (Enfoque Optimizado) : ~{mem_single_month_opt_mb:.2f} MB")
print(f"• Reducción de memoria obtenida                 : ~{((1 - (mem_single_month_opt_mb / mem_naive_mb)) * 100):.1f}%")


RESUMEN DE DESEMPEÑO
• Consumo aproximado por mes (Enfoque Tradicional): ~616.31 MB
• Consumo aproximado por mes (Enfoque Optimizado) : ~105.87 MB
• Reducción de memoria obtenida                 : ~82.8%


# Manipulación

In [15]:
#Establecer una columna como índice
df_opt = df_opt.set_index('tpep_pickup_datetime')

# Ordenar el índice (Práctica recomendada para optimizar búsquedas y slicing)
df_opt = df_opt.sort_index()

# Verificar la estructura resultante
print(df_opt.head())
print("\nTipo de índice:", type(df_opt.index))

                     tpep_dropoff_datetime  passenger_count  trip_distance  \
tpep_pickup_datetime                                                         
2007-12-05 18:45:00    2007-12-05 19:02:00              1.0           3.00   
2009-01-01 00:19:34    2009-01-01 01:10:21              6.0          10.77   
2009-01-01 00:20:39    2009-01-01 00:20:49              5.0           4.47   
2009-01-01 08:52:26    2009-01-01 10:00:26              1.0          11.95   
2009-01-01 12:52:15    2009-01-01 13:12:15              1.0           2.13   

                     PULocationID DOLocationID payment_type  fare_amount  \
tpep_pickup_datetime                                                       
2007-12-05 18:45:00           142          234          2.0    17.000000   
2009-01-01 00:19:34           138          239          2.0    52.700001   
2009-01-01 00:20:39           230          261          1.0    24.000000   
2009-01-01 08:52:26           138          163          1.0    64.599998 

In [16]:
# Columna individual (Retorna una Series)
distancias = df_opt['trip_distance']

# Múltiples columnas (Retorna un DataFrame)
subconjunto = df_opt[['passenger_count', 'fare_amount', 'total_amount']]

# Primeras 3 filas por posición
primeras_filas = df_opt[:3]

# Seleccionar todos los viajes de un día específico
viajes_dia = df_opt.loc['2023-01-15']

# Seleccionar un rango horario preciso
viajes_rango = df_opt.loc['2023-01-15 08:00:00':'2023-01-15 12:00:00']

# Fila específica y columnas seleccionadas
detalle = df_opt.loc['2023-01-15', ['trip_distance', 'total_amount']]

# Rango de filas y rango de columnas
bloque_loc = df_opt.loc['2023-01-15 08:00:00':'2023-01-15 12:00:00', 'trip_distance':'payment_type']

#B. Uso de .iloc (Posiciones numéricas enteras)
# Fila en la posición 0 (primera fila)
primera_posicion = df_opt.iloc[0]

# Filas 0 a 2 y columnas 1 a 3
bloque_iloc = df_opt.iloc[0:3, 1:4]

# Filas específicas y columnas específicas por índice
muestra = df_opt.iloc[[0, 3, 4], [0, 5]]

In [17]:
#3. Filtrado Booleano (Basado en Condiciones)
#En pandas, las condiciones compuestas requieren operadores bit a bit (& para AND, | para OR, ~ para NOT) y paréntesis obligatorios para cada condición.

# Viajes con distancia mayor a 5 millas
viajes_largos = df_opt[df_opt['trip_distance'] > 5.0]

# Viajes pagados con tarjeta (payment_type == 1) Y con más de 1 pasajero
filtro_and = df_opt[(df_opt['payment_type'] == 1) & (df_opt['passenger_count'] > 1)]

# Viajes muy cortos (< 1 milla) O con tarifa alta (> $40)
filtro_or = df_opt[(df_opt['trip_distance'] < 1.0) | (df_opt['fare_amount'] > 40.0)]

# Viajes que NO fueron pagados en efectivo (payment_type != 2)
no_efectivo = df_opt[~(df_opt['payment_type'] == 2)]

In [18]:
#Filtrado por lista de valores (.isin())
# Viajes con origen en las zonas 161 o 237
zonas_interes = [161, 237]
viajes_zonas = df_opt[df_opt['PULocationID'].isin(zonas_interes)]

In [19]:
#Manejo de valores nulos (.isna() / .notna())

# Filtrar filas donde 'passenger_count' NO sea nulo
df_sin_nulos = df_opt[df_opt['passenger_count'].notna()]

# Identificar registros donde falta el conteo de pasajeros
df_con_nulos = df_opt[df_opt['passenger_count'].isna()]

# Procesamiento por lotes

El escáner por lotes de PyArrow (pyarrow.dataset.Scanner / to_batches()) es una de las herramientas fundamentales en la ingeniería de datos moderna para procesar archivos masivos (Big Data) que no caben en la memoria RAM de un solo equipo o máquina virtual.

A diferencia del flujo tradicional donde pandas intenta cargar un archivo gigante de golpe en RAM (```df = pd.read_parquet(...)```), PyArrow implementa una arquitectura de Streaming por Bloques (Chunked Streaming).

- Lectura Paginada / Mapeo de Metadatos: Al instanciar ```dataset.scanner()```, PyArrow solo lee los metadatos y pies de página (footers) del archivo Parquet o Apache Arrow. No carga las filas reales a memoria.

- Proyección en origen: Si especificas filtros o selecciones de columnas (columns=['A', 'B']), PyArrow le ordena al motor de lectura en C++ que ignore físicamente el resto de los datos en el disco.

- Pipelining por Bloques (RecordBatch): La función ```.to_batches()``` fragmenta la lectura. En lugar de devolver un solo objeto gigantesco, entrega un iterador de objetos ```pyarrow.RecordBatch``` (habitualmente de 64,000 a 500,000 filas por lote).

- Liberación continua de RAM: Cada lote pasa por el bucle for, procesa el cálculo (por ejemplo, una suma parcial o un promedio), e inmediatamente la memoria del lote anterior queda libre para ser sobreescrita por el siguiente lote.

In [20]:
print("\n" + "="*60)
print("DEMOSTRACIÓN 3: AGREGACIÓN POR BLOQUES SIN CARGAR TODO EN RAM")
print("="*60)

# Usamos el escáner por lotes de PyArrow para sumar los millones de pasajeros y calcular la cantidad de viajes
total_passengers = 0
total_trips = 0

scanner = dataset.scanner(columns=['passenger_count'], batch_size=500_000)

for batch_idx, batch in enumerate(scanner.to_batches()):
    batch_df = batch.to_pandas()
    passengers = batch_df['passenger_count'].sum()
    trips = len(batch_df)
    
    total_passengers += passengers
    total_trips += trips
    
    if batch_idx < 3 or batch_idx % 5 == 0: #Mostramos resultados para algunos bloques
        print(f"  Bloque {batch_idx+1}: {trips:,} filas procesadas | RAM actual: {psutil.Process(os.getpid()).memory_info().rss / (1024**2):.2f} MB")

print(f"\nResultado final por bloques -> Total de viajes: {total_trips:,} | Promedio de pasajeros: {total_passengers/total_trips:.2f}")


DEMOSTRACIÓN 3: AGREGACIÓN POR BLOQUES SIN CARGAR TODO EN RAM
  Bloque 1: 500,000 filas procesadas | RAM actual: 16196.17 MB
  Bloque 2: 500,000 filas procesadas | RAM actual: 16245.41 MB
  Bloque 3: 48,576 filas procesadas | RAM actual: 16251.40 MB
  Bloque 6: 48,576 filas procesadas | RAM actual: 16208.69 MB
  Bloque 11: 500,000 filas procesadas | RAM actual: 16218.16 MB
  Bloque 16: 48,576 filas procesadas | RAM actual: 16224.17 MB
  Bloque 21: 500,000 filas procesadas | RAM actual: 16229.93 MB
  Bloque 26: 48,576 filas procesadas | RAM actual: 16233.99 MB
  Bloque 31: 499,529 filas procesadas | RAM actual: 16239.18 MB
  Bloque 36: 500,000 filas procesadas | RAM actual: 16243.31 MB
  Bloque 41: 500,000 filas procesadas | RAM actual: 16244.99 MB
  Bloque 46: 500,000 filas procesadas | RAM actual: 16247.36 MB
  Bloque 51: 48,576 filas procesadas | RAM actual: 16250.17 MB
  Bloque 56: 500,000 filas procesadas | RAM actual: 16252.35 MB
  Bloque 61: 48,576 filas procesadas | RAM actual: